# Keras CIFAR-10 run on Colab

Runs the repository's own `train.py` unmodified, on the same 15,000-image split the
PyTorch baseline used, and checks the model has **815,018 trainable parameters**.

The split is not uploaded as images. `data/cifar10_split_indices.json` holds the exact
indices the PyTorch run selected (`torch.randperm(50000, seed=42)[:15000]`, then a
13,500 / 1,500 split with the same seed). CIFAR-10 itself is downloaded by Keras, so the
only thing that travels is the index list.

## Files to upload (3)

| Upload this | It came from |
|---|---|
| `train.py` | repo root |
| `model.py` | repo root |
| `cifar10_split_indices.json` | repo `data/` folder |

Upload all three in the one upload box in step 2. The notebook puts them in the right
folders for you.

## Step 1 - what Colab gave us

Records the versions so the result file says what it actually ran on. This repo's
`Dockerfile` pins `tensorflow-cpu==2.20.0` and `numpy==2.1.3`; Colab's preinstalled
versions are usually different. That changes nothing about the parameter count or the
split, but it can move the final accuracy by a few tenths, so it is recorded rather than
hidden.

In [ ]:
import platform, sys

import numpy as np
import tensorflow as tf
import keras

print("python     ", platform.python_version())
print("tensorflow ", tf.__version__)
print("keras      ", keras.__version__)
print("numpy      ", np.__version__)
print("GPUs       ", tf.config.list_physical_devices("GPU") or "none (CPU only)")

## Step 2 - upload the three files

Run the cell, click **Choose Files**, and select `train.py`, `model.py` and
`cifar10_split_indices.json` together. The cell then moves them into the layout
`train.py` expects:

```
/content/work/train.py
/content/work/model.py
/content/work/data/cifar10_split_indices.json
```

In [ ]:
import shutil
from pathlib import Path

from google.colab import files

WORK = Path("/content/work")
(WORK / "data").mkdir(parents=True, exist_ok=True)

uploaded = files.upload()

DESTINATIONS = {
    "train.py": WORK / "train.py",
    "model.py": WORK / "model.py",
    "cifar10_split_indices.json": WORK / "data" / "cifar10_split_indices.json",
}

for name in uploaded:
    target = DESTINATIONS.get(Path(name).name)
    if target is None:
        print(f"ignored (not one of the three expected files): {name}")
        continue
    shutil.move(name, target)
    print(f"placed {name} -> {target}")

missing = [str(p.relative_to(WORK)) for p in DESTINATIONS.values() if not p.exists()]
if missing:
    raise SystemExit(
        "still missing: " + ", ".join(missing) + "\nRe-run this cell and select them too."
    )
print("\nall three files present")

## Step 3 - check the split and the parameter count *before* training

This is the assertion the notebook exists to make, and it is checked here rather than
only inside `train.py` so a failure costs seconds instead of the whole run.

Three things are asserted:

1. the split is **13,500 train + 1,500 val = 15,000**, and the two index sets do not overlap
2. the uploaded JSON records `subset_size == 15000`
3. the Keras model has exactly **815,018** trainable parameters

`train.py` asserts the parameter count again on its own (it exits non-zero on a
mismatch), so the number is checked twice by two independent code paths.

In [ ]:
import json
import sys

sys.path.insert(0, "/content/work")

import model as model_module

EXPECTED_PARAMETERS = 815_018
EXPECTED_TRAIN, EXPECTED_VAL = 13_500, 1_500

split = json.loads(
    (WORK / "data" / "cifar10_split_indices.json").read_text(encoding="utf-8")
)
train_idx, val_idx = split["train_indices"], split["val_indices"]

print(f"train indices      {len(train_idx):,}")
print(f"val indices        {len(val_idx):,}")
print(f"total              {len(train_idx) + len(val_idx):,}")
print(f"subset_size field  {split['subset_size']:,}")
print(f"exported with      torch {split['torch_version']}")
print(f"source             {split['source']}")

assert len(train_idx) == EXPECTED_TRAIN, f"train is {len(train_idx)}, expected {EXPECTED_TRAIN}"
assert len(val_idx) == EXPECTED_VAL, f"val is {len(val_idx)}, expected {EXPECTED_VAL}"
assert len(train_idx) + len(val_idx) == 15_000
assert not (set(train_idx) & set(val_idx)), "train and val indices overlap"
assert split["subset_size"] == 15_000
print("\nsplit OK: 13,500 + 1,500 = 15,000, no overlap")

net = model_module.build_model(dropout=0.25)
trainable = model_module.count_trainable(net)
print(f"\ntrainable parameters {trainable:,}")
assert trainable == EXPECTED_PARAMETERS, f"{trainable} != {EXPECTED_PARAMETERS}"
print(f"parameter count OK: exactly {EXPECTED_PARAMETERS:,}")

net.summary()

## Step 4 - train

Runs `train.py` exactly as the repo's README does, with no edits:

```
python train.py --out results/keras_results.json
```

12 epochs over 13,500 images. Expect roughly 5-15 minutes: the augmentation in
`augment_batch` is a per-image NumPy loop on the CPU, so a GPU runtime helps the
forward/backward pass but not that loop.

The PyTorch baseline this is compared against reached **64.26%** test accuracy.

In [ ]:
%cd /content/work
!python train.py --out results/keras_results.json

## Step 5 - show the result, and re-check the number in the saved file

The parameter count is read back out of the written JSON, so the proof survives in the
file and not just in the scrollback.

In [ ]:
import json
from pathlib import Path

record = json.loads(Path("/content/work/results/keras_results.json").read_text(encoding="utf-8"))

print(json.dumps(record, indent=2))

assert record["n_parameters"] == 815_018, record["n_parameters"]
assert record["data"]["train"] == 13_500
assert record["data"]["val"] == 1_500
assert record["data"]["train"] + record["data"]["val"] == 15_000

print("\n--- checks on the saved file ---")
print(f"n_parameters        {record['n_parameters']:,}  (expected 815,018)")
print(f"train + val         {record['data']['train']:,} + {record['data']['val']:,}")
print(f"keras test accuracy {record['test_accuracy']:.2f}%")
print(f"pytorch baseline    64.26%")
print(f"difference          {record['test_accuracy'] - 64.26:+.2f} points")

## Step 6 - download the result file

Saves `keras_results.json` to your computer. Send me that file plus this notebook
(**File -> Download -> Download .ipynb**, after the cells have run so the outputs are
saved in it) and I will commit both.

In [ ]:
from google.colab import files

files.download("/content/work/results/keras_results.json")